# Pyomo and jaxglobal, side by side

This series writes the same optimization problems twice: once as a [Pyomo](https://www.pyomo.org) model [hart2011pyomo, bynum2021pyomo], and once as a plain Python/JAX function for jaxglobal,
a global optimizer that traces the function and bounds it with McCormick/alphaBB relaxations inside a branch-and-bound [mccormick1976computability]. Every notebook solves both and
compares objective, certified bound, time and status. The audience is chemical engineers and optimization researchers who already know one of the two styles.

The comparison is meant to be fair. Pyomo has a large ecosystem, mature global solvers (SCIP, BARON, Couenne here), logical (GDP) modelling, duals,
named constraints and sparse large-scale structure. jaxglobal is a research solver for continuous and integer variables; these notebooks use only continuous ones. Where Pyomo is faster or more capable, the notebooks say so.

| Notebook | Topic |
|---|---|
| 01-basics | Six-hump camel function; Haverly pooling problem |
| 02-control-flow | `where`, `clip`, `max`, `interp`: pipe flow regimes and steam tables; Pyomo GDP and `Piecewise` |
| 03-ode-models | Kinetic parameter estimation with an ODE; explicit RK4 and `pyomo.dae` |
| 04-neural-network-surrogates | Minimizing a trained tanh MLP; hand-written Pyomo and OMLT |
| 05-things-pyomo-cannot-express | `fori_loop`, library functions, einsum, parametric families |

**Conditions.** All solvers use relative gap 1e-4 (absolute 1e-6) and at most 60 s per solve. Timings are wall-clock on a loaded laptop and include
the model-writing time for Pyomo solvers and JIT compilation for jaxglobal; treat them as rough. All data are either published problem data or synthetic (labeled in each notebook).

**Reading the tables.** `obj` is the best objective found, `bound` the certified lower bound the solver reports (NaN means the solver did not return one;
Couenne does not through the `.nl` interface), `status` the solver's termination.

**References.** Citation keys in square brackets refer to entries in `docs/references.bib` at the repository root.

## Counting lines of model code

"Lines of model code" is measured by one helper, `nbutil.loc`, which takes the functions that make up a model and counts source lines excluding blanks, comments and docstrings.
Its source:

In [1]:
import inspect
import json
import os

import numpy as np
import pandas as pd

import nbutil
from nbutil import loc

print(inspect.getsource(loc))

def loc(*objs):
    """Count lines of model code: source lines minus blank lines, comments and docstrings."""
    n = 0
    for o in objs:
        src = inspect.getsource(o)
        src = re.sub(r'("""|\'\'\')(.*?)\1', "", src, flags=re.S)  # drop docstrings
        n += sum(1 for ln in src.splitlines() if ln.strip() and not ln.strip().startswith("#"))
    return n



## Results of all comparisons

Filled in from the results saved by notebooks 01 to 05 (`notebooks/results/*.json`). Run those notebooks first.
Model size is the Pyomo model's unfixed variables and active constraints (the jaxglobal model has no explicit constraints beyond optional `ineq`/`eq` functions).

In [2]:
here = os.path.dirname(os.path.abspath(nbutil.__file__))
NAMES = {"01": "01-basics", "02": "02-control-flow", "03": "03-ode-models", "04": "04-neural-network-surrogates", "05": "05-things-pyomo-cannot-express"}
cases = []
for nb in sorted(NAMES):
    path = os.path.join(here, "results", f"{nb}.json")
    if os.path.exists(path):
        for case, d in json.load(open(path)).items():
            cases.append((NAMES[nb], case, d))
print(f"{len(cases)} comparisons from {len({c[0] for c in cases})} notebooks")

12 comparisons from 5 notebooks


In [3]:
size_rows = [
    {
        "notebook": nb,
        "case": case,
        "loc Pyomo": d["loc_pyomo"],
        "loc jaxglobal": d["loc_jax"],
        "Pyomo vars": d["pyomo_vars"],
        "Pyomo constraints": d["pyomo_cons"],
    }
    for nb, case, d in cases
]
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)
pd.DataFrame(size_rows).fillna("n/a")

,notebook,case,loc Pyomo,loc jaxglobal,Pyomo vars,Pyomo constraints
0,01-basics,six-hump camel,8.0,3.0,2.0,0.0
1,01-basics,Haverly pooling,19.0,9.0,9.0,6.0
2,02-control-flow,"A: pipe (where, clip)",28.0,11.0,14.0,16.0
3,02-control-flow,"B: steam tables (interp, max)",15.0,11.0,33.0,32.0
4,03-ode-models,RK4 kinetics (explicit),24.0,14.0,10.0,8.0
5,03-ode-models,pyomo.dae (collocation),16.0,14.0,52.0,50.0
6,04-neural-network-surrogates,tanh MLP 2-8-8-1 (reduced space),11.0,2.0,2.0,0.0
7,04-neural-network-surrogates,tanh MLP 2-8-8-1 (full space),19.0,2.0,34.0,32.0
8,05-things-pyomo-cannot-express,Colebrook fori_loop,11.0,10.0,3.0,1.0
9,05-things-pyomo-cannot-express,library functions,n/a,n/a,n/a,n/a


Pyomo needs more lines in most cases, partly because it must declare variables, bounds and constraints that a function does not need. The line count says little about
solver quality, and model sizes are for the Pyomo formulation only.

In [4]:
res_rows = []
for nb, case, d in cases:
    for r in d["rows"]:
        res_rows.append({"notebook": nb[:2], "case": case, "solver": r["solver"], "obj": r["obj"], "bound": r["bound"], "time (s)": round(r["time"], 2) if r["time"] == r["time"] else np.nan, "status": r["status"]})
df = pd.DataFrame(res_rows)
df.style.format({"obj": "{:.6g}", "bound": "{:.6g}"}, na_rep="n/a").hide(axis="index")

notebook,case,solver,obj,bound,time (s),status
01,six-hump camel,scip,-1.03163,-1.03171,0.170000,optimal
01,six-hump camel,couenne,-1.03163,n/a,0.290000,optimal
01,six-hump camel,baron,-1.03163,-1.03163,0.960000,optimal
01,six-hump camel,jaxglobal,-1.03163,-1.03169,5.150000,optimal
01,six-hump camel,"jaxglobal (2nd call, cached)",-1.03163,-1.03169,0.030000,optimal
01,Haverly pooling,scip,-400,-400,0.070000,optimal
01,Haverly pooling,couenne,-400,n/a,0.370000,optimal
01,Haverly pooling,baron,-400,-400,0.840000,optimal
01,Haverly pooling,jaxglobal,-400,-400.014,12.010000,optimal
02,"A: pipe (where, clip)",scip,-2828.74,n/a,0.290000,optimal


## Summary

* **Speed.** On these small problems the Pyomo solvers (SCIP, BARON) finish in about a second or a few seconds. jaxglobal takes from a few seconds to about 30 s, a large part of which is JIT compilation of the traced function.
  Where the same function is solved again, the second solve is much faster (notebook 01).
* **Agreement.** Where both sides finish, objectives agree to the gap tolerance.
* **Where jaxglobal reads better.** Models that are numerical code: ODE integrators, neural networks, loops, library functions, vectorized algebra. There is no transcription step.
* **Where Pyomo is better.** Integer and logical structure, a choice of mature solvers, named constraints and duals, sparse large-scale models, and a large ecosystem (`pyomo.dae`, GDP, OMLT).
* **Current jaxglobal limits found while writing these notebooks.** Compile time grows quickly with the length of an unrolled loop (notebook 03); `lax.scan` and `jnp.interp` are not supported
  (notebooks 02, 03). `jax.scipy.special.erfc`, and with it the normal CDF, now has a rule (notebook 05).